# Mini-Lab A — Custom Prediction Routine (CPR)

**Target Gap:** Abstraction level selection (serving architecture)  
**Dataset:** Census income (same as Labs 1–5)  
**Task:** Serve a GradientBoostingClassifier using CPR instead of a custom Flask container  
**Exam Relevance:** Very high — "custom model + preprocessing + minimize code" → CPR

---

### What You'll Learn
1. How CPR replaces the Flask serving container you built in Lab 2
2. The `SklearnPredictor` interface: load, preprocess, predict, postprocess
3. `LocalModel.build_cpr_model()` — auto-generated Dockerfile and model server
4. Side-by-side comparison: what CPR eliminates vs what it preserves

### Parts
| Part | Focus | Time |
|------|-------|------|
| 1 | Setup & Model Training | ~20 min |
| 2 | Build the CPR Predictor | ~30 min |
| 3 | Build CPR Container, Deploy & Test | ~20 min |
| 4 | Comparison, Reflection & Cleanup | ~15 min |

### Estimated Cost: $1–2

### Context: Lab 2 Recap

In Lab 2, you built a **custom Flask container** for serving. That container included:
- A Flask HTTP server with `/predict` and `/health` routes (~45 lines in `serve.py`)
- Manual model loading via `gsutil cp` + `joblib.load()`
- JSON request parsing → DataFrame → inference → JSON response
- A hand-written Dockerfile (`python:3.10-slim` + sklearn + Flask)

CPR eliminates the HTTP server, routing, health checks, and Dockerfile — you only write the ML logic.

---
## Part 1: Setup & Model Training

We need a trained model artifact in GCS before we can serve it. Since this is a standalone lab
(no dependencies on previous lab infrastructure), we'll rebuild a minimal custom training container
and run a quick training job on Vertex AI.

**Container decision:** Custom `python:3.10-slim` for training (same proven pattern from Lab 2).
CPR will generate the serving container for us — that's the whole point of Part 2.

### 1.1 Install Dependencies

In [ ]:
# CPR requires the prediction extras from the Vertex AI SDK
!pip install --quiet google-cloud-aiplatform[prediction] google-cloud-storage google-cloud-bigquery

### 1.2 Configuration

In [2]:
import os
from datetime import datetime

# --- Project Configuration ---
PROJECT_ID = "carty-470812"
REGION = "us-central1"
BUCKET_NAME = PROJECT_ID
BUCKET_URI = f"gs://{BUCKET_NAME}"

# Artifact Registry
AR_REPO = "ml-containers"
TRAIN_IMAGE_NAME = "census-train-minilab-a"
TRAIN_IMAGE_URI = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{AR_REPO}/{TRAIN_IMAGE_NAME}:latest"

# CPR serving image (will be built by CPR in Part 3)
CPR_IMAGE_NAME = "census-cpr"
CPR_IMAGE_URI = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{AR_REPO}/{CPR_IMAGE_NAME}:latest"

# GCS paths
DATA_PATH = f"{BUCKET_URI}/mini-lab-a/data/census_income.csv"
MODEL_DIR = f"{BUCKET_URI}/mini-lab-a/model/"

# Timestamp for unique resource naming
TIMESTAMP = datetime.now().strftime("%Y%m%d-%H%M%S")

print(f"Project:         {PROJECT_ID}")
print(f"Region:          {REGION}")
print(f"Train Image:     {TRAIN_IMAGE_URI}")
print(f"CPR Image:       {CPR_IMAGE_URI}")
print(f"Data Path:       {DATA_PATH}")
print(f"Model Dir:       {MODEL_DIR}")
print(f"Timestamp:       {TIMESTAMP}")

Project:         carty-470812
Region:          us-central1
Train Image:     us-central1-docker.pkg.dev/carty-470812/ml-containers/census-train-minilab-a:latest
CPR Image:       us-central1-docker.pkg.dev/carty-470812/ml-containers/census-cpr:latest
Data Path:       gs://carty-470812/mini-lab-a/data/census_income.csv
Model Dir:       gs://carty-470812/mini-lab-a/model/
Timestamp:       20260308-175839


### 1.3 Initialize Clients

In [3]:
from google.cloud import aiplatform, bigquery, storage

aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=BUCKET_URI)
bq_client = bigquery.Client(project=PROJECT_ID)
storage_client = storage.Client(project=PROJECT_ID)

print("✅ Clients initialized")

✅ Clients initialized


### 1.4 Create GCS Bucket & Artifact Registry Repo (if needed)

In [4]:
# Create bucket if it doesn't exist
!gsutil ls gs://{BUCKET_NAME} 2>/dev/null || gsutil mb -l {REGION} gs://{BUCKET_NAME}

# Create Artifact Registry repository if it doesn't exist
!gcloud artifacts repositories describe {AR_REPO} --location={REGION} 2>/dev/null ||     gcloud artifacts repositories create {AR_REPO}     --repository-format=docker     --location={REGION}     --description="ML training and serving containers"

# Configure Docker authentication for Artifact Registry
!gcloud auth configure-docker {REGION}-docker.pkg.dev --quiet

print("✅ Infrastructure ready")

gs://carty-470812/aiplatform-2026-03-08-10:57:35.625-aiplatform_custom_trainer_script-0.1.tar.gz
gs://carty-470812/aiplatform-2026-03-08-11:06:12.640-aiplatform_custom_trainer_script-0.1.tar.gz
gs://carty-470812/aiplatform-2026-03-08-11:11:57.482-aiplatform_custom_trainer_script-0.1.tar.gz
gs://carty-470812/aiplatform-2026-03-08-11:54:33.571-aiplatform_custom_trainer_script-0.1.tar.gz
gs://carty-470812/aiplatform-2026-03-08-14:00:08.580-aiplatform_custom_trainer_script-0.1.tar.gz
gs://carty-470812/aiplatform-custom-training-2026-03-08-11:54:33.696/
gs://carty-470812/aiplatform-custom-training-2026-03-08-14:00:08.692/
gs://carty-470812/lab10-pipelines/
gs://carty-470812/mini-lab-a/
createTime: '2026-03-08T16:15:45.382519Z'
description: ML training and serving containers
format: DOCKER
mode: STANDARD_REPOSITORY
name: projects/carty-470812/locations/us-central1/repositories/ml-containers
registryUri: us-central1-docker.pkg.dev/carty-470812/ml-containers
satisfiesPzi: true
updateTime: '202

### 1.5 Export Census Data to GCS

Query from BigQuery and upload to GCS. Same dataset as Labs 1–5.

In [5]:
# Query census data from BigQuery
query = """
SELECT
    age, workclass, education, education_num, marital_status,
    occupation, relationship, race, sex, capital_gain,
    capital_loss, hours_per_week, native_country, income_bracket
FROM `bigquery-public-data.ml_datasets.census_adult_income`
"""

df = bq_client.query(query).to_dataframe()
print(f"Loaded {len(df):,} rows from BigQuery")
print(f"Target distribution:\n{df['income_bracket'].value_counts()}")

# Save to GCS
df.to_csv(f"/tmp/census_income.csv", index=False)
!gsutil cp /tmp/census_income.csv {DATA_PATH}
print(f"\n✅ Data uploaded to {DATA_PATH}")

/Users/james.carty/Documents/VScode/google-ml-engineer/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Loaded 32,561 rows from BigQuery
Target distribution:
income_bracket
<=50K    24720
>50K      7841
Name: count, dtype: int64
Copying file:///tmp/census_income.csv [Content-Type=text/csv]...
\ [1 files][  3.4 MiB/  3.4 MiB]                                                
Operation completed over 1 objects/3.4 MiB.                                      

✅ Data uploaded to gs://carty-470812/mini-lab-a/data/census_income.csv


### 1.6 Write Training Script

Minimal training script — GradientBoostingClassifier with Lab 3's best hyperparameters.
The only job of this script is to produce a `model.joblib` artifact in GCS.

In [29]:
%%writefile train.py
"""
Minimal training script for Mini-Lab A.
Produces a model.joblib artifact compatible with CPR's SklearnPredictor.
"""

import argparse
import os
import pandas as pd
import joblib
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--data-path', type=str, required=True)
    parser.add_argument('--model-dir', type=str, default=os.environ.get('AIP_MODEL_DIR', '/tmp/model'))
    args = parser.parse_args()

    # Load data
    print(f"Loading data from: {args.data_path}")
    df = pd.read_csv(args.data_path)
    print(f"Loaded {len(df):,} rows")

    # Prepare features and target
    target = 'income_bracket'
    X = df.drop(columns=[target])
    y = (df[target].str.strip() == '>50K').astype(int)

    # One-hot encode categoricals
    X = pd.get_dummies(X, drop_first=True)

    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")

    # Train with Lab 3 best hyperparameters
    model = GradientBoostingClassifier(
        n_estimators=385,
        max_depth=9,
        learning_rate=0.028,
        min_samples_split=5,
        random_state=42
    )
    model.fit(X_train, y_train)

    # Evaluate
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"\nAccuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(classification_report(y_test, y_pred, target_names=['<=50K', '>50K']))

    # Save model artifact locally first, then upload to GCS
    local_dir = '/tmp/model_output'
    os.makedirs(local_dir, exist_ok=True)
    
    model_path = os.path.join(local_dir, 'model.joblib')
    joblib.dump(model, model_path)
    print(f"✅ Model saved locally: {model_path}")
    
    feature_path = os.path.join(local_dir, 'feature_columns.joblib')
    joblib.dump(list(X_train.columns), feature_path)
    print(f"✅ Feature columns saved locally: {feature_path}")
    
    # Upload to GCS using google-cloud-storage
    from google.cloud import storage
    
    # Parse bucket and prefix from gs:// path
    gcs_path = args.model_dir.replace('gs://', '')
    bucket_name = gcs_path.split('/')[0]
    prefix = '/'.join(gcs_path.split('/')[1:]).rstrip('/')
    
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    
    for filename in ['model.joblib', 'feature_columns.joblib']:
        blob = bucket.blob(f"{prefix}/{filename}")
        blob.upload_from_filename(os.path.join(local_dir, filename))
        print(f"✅ Uploaded {filename} to gs://{bucket_name}/{prefix}/{filename}")

if __name__ == '__main__':
    main()

Overwriting train.py


### 1.7 Build & Push Training Container

Minimal Dockerfile — just enough to run the training script on Vertex AI.

In [30]:
%%writefile Dockerfile
FROM python:3.10-slim

WORKDIR /app

RUN pip install --no-cache-dir \
    scikit-learn==1.3.2 \
    pandas==2.1.4 \
    numpy==1.26.2 \
    joblib==1.3.2 \
    google-cloud-storage==2.14.0 \
    gcsfs>=2023.12.0

COPY train.py /app/train.py

ENTRYPOINT ["python", "train.py"]

Overwriting Dockerfile


In [31]:
# Build and push training container
!docker build --platform linux/amd64 -t {TRAIN_IMAGE_URI} .
!docker push {TRAIN_IMAGE_URI}

print(f"\n✅ Training image pushed: {TRAIN_IMAGE_URI}")



[+] Building 0.0s (0/1)                                    docker:desktop-linux
[+] Building 0.2s (1/2)                                    docker:desktop-linux
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 315B                                       0.0s
 => [internal] load metadata for docker.io/library/python:3.10-slim        0.1s
[+] Building 0.3s (1/2)                                    docker:desktop-linux
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 315B                                       0.0s
 => [internal] load metadata for docker.io/library/python:3.10-slim        0.3s
[+] Building 0.5s (1/2)                                    docker:desktop-linux
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 315B                                       0.0s
 => [internal] load metadata for docke

### 1.8 Submit Training Job

In [32]:
# Submit training job to Vertex AI
# Using CustomJob (not CustomContainerTrainingJob) because we don't have
# the serving container yet — CPR will build it in Part 3.
job = aiplatform.CustomJob(
    display_name=f"census-minilab-a-train-{TIMESTAMP}",
    worker_pool_specs=[{
        "machine_spec": {"machine_type": "n1-standard-4"},
        "replica_count": 1,
        "container_spec": {
            "image_uri": TRAIN_IMAGE_URI,
            "args": ["--data-path", DATA_PATH, "--model-dir", MODEL_DIR],
        },
    }],
    staging_bucket=BUCKET_URI,
)

job.run()

print(f"\n✅ Training complete. Model artifact at: {MODEL_DIR}")


Creating CustomJob
CustomJob created. Resource name: projects/873708835509/locations/us-central1/customJobs/9115855313080156160
To use this CustomJob in another session:
custom_job = aiplatform.CustomJob.get('projects/873708835509/locations/us-central1/customJobs/9115855313080156160')
View Custom Job:
https://console.cloud.google.com/ai/platform/locations/us-central1/training/9115855313080156160?project=873708835509
CustomJob projects/873708835509/locations/us-central1/customJobs/9115855313080156160 current state:
2
CustomJob projects/873708835509/locations/us-central1/customJobs/9115855313080156160 current state:
2
CustomJob projects/873708835509/locations/us-central1/customJobs/9115855313080156160 current state:
2
CustomJob projects/873708835509/locations/us-central1/customJobs/9115855313080156160 current state:
2
CustomJob projects/873708835509/locations/us-central1/customJobs/9115855313080156160 current state:
2
CustomJob projects/873708835509/locations/us-central1/customJobs/91158

> **Why `CustomJob` instead of `CustomContainerTrainingJob`?**
>
> `CustomContainerTrainingJob` requires a serving container image at submission time,
> but we haven't built the CPR container yet (that's Part 3). `CustomJob` just runs
> the training script and saves the artifact to GCS — no serving image needed.
> We'll connect the model artifact to the CPR serving container in Part 3 via `Model.upload()`.


---
## Part 2: Build the CPR Predictor

This is the core of the lab. You'll write a `CensusPredictor` class that replaces your entire
Lab 2 `serve.py` — Flask server, health checks, request routing, and all.

### What CPR gives you for free
- HTTP server (replaces Flask)
- Request routing (replaces `/predict` and `/health` routes)
- Health check endpoint (replaces your `health()` function)
- Request deserialization (replaces `request.get_json()`)
- Response serialization (replaces `jsonify()`)
- Dockerfile generation (replaces your hand-written Dockerfile)

### What you write
- `load()` — how to load your model from GCS artifacts
- `preprocess()` — transform raw input into model-ready format
- `predict()` — run inference
- `postprocess()` — format the output

### Lab 2 serve.py vs CPR Predictor

```
Lab 2 serve.py (~45 lines)                    CPR predictor.py (~35 lines)
─────────────────────────                      ──────────────────────────────
from flask import Flask, ...                   from ...sklearn.predictor import SklearnPredictor
app = Flask(__name__)                          
model = None                                   class CensusPredictor(SklearnPredictor):
                                               
def load_model():                                  def load(self, artifacts_uri):
    storage_uri = os.environ.get(...)                  super().load(artifacts_uri)
    gsutil cp ...                                      # load feature columns
    model = joblib.load(...)                   
                                               
@app.route('/health')                              # ← FREE (auto-generated)
def health(): ...                              
                                               
@app.route('/predict')                             def preprocess(self, prediction_input):
def predict():                                         # validate & transform input
    content = request.get_json()               
    instances = content.get('instances')        
    df = pd.DataFrame(instances)                   def predict(self, instances):
    predictions = model.predict(df)                    # inherited from SklearnPredictor
    return jsonify({'predictions': ...})       
                                                   def postprocess(self, prediction_results):
if __name__ == '__main__':                             # format output with probabilities
    load_model()                               
    app.run(host='0.0.0.0', port=...)          # ← FREE (auto-generated)
```

### 2.1 Create Source Directory

CPR expects a source directory with your predictor and requirements file.

In [45]:
import os

# Create CPR source directory
CPR_SRC_DIR = "cpr_src"
os.makedirs(CPR_SRC_DIR, exist_ok=True)
print(f"✅ Created {CPR_SRC_DIR}/")

✅ Created cpr_src/


### 2.2 Write requirements.txt

These are the dependencies for the **serving** container — not the training container.
Only include what the predictor needs at inference time.

In [36]:
%%writefile cpr_src/requirements.txt
# Serving dependencies for CPR container
# These must be compatible with the base image (python:3.10-slim)
scikit-learn==1.3.2
pandas==2.1.4
numpy==1.26.2
joblib==1.3.2
google-cloud-storage==2.14.0
google-cloud-aiplatform[prediction]

Overwriting cpr_src/requirements.txt


### 2.3 Write the CensusPredictor

This is the heart of the lab. We subclass `SklearnPredictor` to get free model loading,
then customize preprocessing and postprocessing.

**Why `SklearnPredictor` and not the raw `Predictor`?**
- `SklearnPredictor` inherits from `Predictor` and adds sklearn-specific model loading
- It calls `joblib.load()` on the model artifact automatically in its default `load()`
- We override `load()` to also load our feature columns, then call `super().load()` for the model
- Less code, same result — and we override the base image to `python:3.10-slim` to avoid the 3.7 default

In [57]:
%%writefile cpr_src/predictor.py
"""
Custom Prediction Routine for census income classification.

Replaces the Lab 2 Flask serve.py with a structured Predictor class.
Vertex AI handles HTTP server, routing, health checks — we only write ML logic.
"""

import os
import joblib
import numpy as np
import pandas as pd
from google.cloud.aiplatform.prediction.sklearn.predictor import SklearnPredictor


class CensusPredictor(SklearnPredictor):
    """Census income predictor using CPR.

    Compared to Lab 2's serve.py:
    - No Flask, no routes, no health endpoint — Vertex AI handles all of that
    - Same preprocessing logic, structured into dedicated methods
    - Postprocessing adds class labels and probabilities (Lab 2 only returned raw predictions)
    """

    def __init__(self):
        super().__init__()
        self._feature_columns = None

    def load(self, artifacts_uri: str) -> None:
        """Load model and feature columns from GCS.

        SklearnPredictor.load() handles joblib model loading.
        We extend it to also load the feature column list saved during training.
        """
        super().load(artifacts_uri)
        
        # artifacts_uri could be GCS or local — download_model_artifacts handles both
        from google.cloud.aiplatform.utils import prediction_utils
        prediction_utils.download_model_artifacts(artifacts_uri)
        
        # After download, files are in the current working directory
        self._feature_columns = joblib.load("feature_columns.joblib")
        print(f"✅ Loaded {len(self._feature_columns)} feature columns")

    def preprocess(self, prediction_input: dict) -> pd.DataFrame:
        """Transform raw JSON input into model-ready DataFrame.

        In Lab 2's serve.py, this was embedded inside the /predict route:
            content = request.get_json()
            instances = content.get('instances', [])
            df = pd.DataFrame(instances)

        Here it's a dedicated method with proper validation and encoding.

        Args:
            prediction_input: Dict with 'instances' key containing list of dicts.
                Each dict has raw feature names/values (before one-hot encoding).

        Returns:
            DataFrame with one-hot encoded features matching training columns.
        """
        # Extract instances from the prediction input
        instances = prediction_input.get("instances", prediction_input)

        # Convert to DataFrame
        df = pd.DataFrame(instances)

        # One-hot encode categorical features (same as training)
        df = pd.get_dummies(df, drop_first=True)

        # Align columns with training features
        # - Add missing columns (set to 0) — handles unseen categories
        # - Remove extra columns — handles categories not in training data
        df = df.reindex(columns=self._feature_columns, fill_value=0)

        return df

    def predict(self, instances: pd.DataFrame) -> np.ndarray:
        """Run inference. Inherited from SklearnPredictor — returns model.predict().

        We override to return predict_proba instead, giving us class probabilities
        that postprocess() can format nicely.
        """
        return self._model.predict_proba(instances)

    def postprocess(self, prediction_results: np.ndarray) -> dict:
        """Format prediction output with class labels and probabilities.

        In Lab 2's serve.py, this was just:
            return jsonify({'predictions': predictions})

        Here we add structured output with class labels and confidence scores.

        Args:
            prediction_results: Array of shape (n_samples, 2) with class probabilities.

        Returns:
            Dict with predictions in Vertex AI's expected format.
        """
        predictions = []
        for probs in prediction_results:
            predicted_class = int(np.argmax(probs))
            predictions.append({
                "predicted_label": ">50K" if predicted_class == 1 else "<=50K",
                "confidence": float(probs[predicted_class]),
                "probabilities": {
                    "<=50K": float(probs[0]),
                    ">50K": float(probs[1]),
                },
            })

        return {"predictions": predictions}

Overwriting cpr_src/predictor.py


### 2.4 Review What You Wrote vs What Lab 2 Required

Take a moment to compare. The predictor above is ~70 lines including docstrings and comments.
Your Lab 2 `serve.py` was ~45 lines — but it also needed a hand-written Dockerfile.

**What's gone:**
- Flask import and app setup
- `/health` endpoint (auto-generated by CPR)
- `/predict` route decorator and request parsing
- `jsonify()` response wrapping
- `app.run()` server startup
- Manual `gsutil cp` for model download

**What's new:**
- Structured `preprocess()` → `predict()` → `postprocess()` pipeline
- Column alignment with `reindex()` (handles unseen categories gracefully)
- Rich output format (labels + probabilities + confidence)

**What stayed the same:**
- `pd.DataFrame(instances)` — converting input to DataFrame
- `model.predict_proba()` — the actual inference call
- `joblib.load()` — model deserialization (handled by `SklearnPredictor.load()`)

---
## Part 3: Build CPR Container, Deploy & Test

Now we use CPR's SDK tooling to auto-generate a Dockerfile, build the container image,
push it to Artifact Registry, and deploy to an endpoint.

### 3.1 Build the CPR Container Image

`LocalModel.build_cpr_model()` does three things:
1. Generates a Dockerfile based on your predictor, requirements, and base image
2. Builds the Docker image locally
3. Returns a `LocalModel` object you can push and deploy

**Key:** We pass `base_image="python:3.10-slim"` to override CPR's default `python:3.7` base.
This ensures the serving container's Python version matches the training container, avoiding
the serialization mismatch issues we've hit in previous labs.

In [58]:
from google.cloud.aiplatform.prediction import LocalModel
from cpr_src.predictor import CensusPredictor

# Need to be able to import from our source directory
import sys
sys.path.insert(0, ".")

local_model = LocalModel.build_cpr_model(
    CPR_SRC_DIR,
    CPR_IMAGE_URI,
    predictor=CensusPredictor,
    requirements_path=os.path.join(CPR_SRC_DIR, "requirements.txt"),
    base_image="python:3.10-slim",  # Override default python:3.7
    platform="linux/amd64",  # Force x86 build for Vertex AI
)

print(f"\n✅ CPR container built: {CPR_IMAGE_URI}")

/opt/homebrew/Cellar/python@3.13/3.13.5/Frameworks/Python.framework/Versions/3.13/lib/python3.13/subprocess.py:1023: RuntimeWarning: line buffering (buffering=1) isn't supported in binary mode, the default buffer size will be used
  self.stdin = io.open(p2cwrite, 'wb', bufsize)
/opt/homebrew/Cellar/python@3.13/3.13.5/Frameworks/Python.framework/Versions/3.13/lib/python3.13/subprocess.py:1029: RuntimeWarning: line buffering (buffering=1) isn't supported in binary mode, the default buffer size will be used
  self.stdout = io.open(c2pread, 'rb', bufsize)



✅ CPR container built: us-central1-docker.pkg.dev/carty-470812/ml-containers/census-cpr:latest


### 3.2 Inspect the Generated Dockerfile

This is the Dockerfile that CPR auto-generated. Compare it to your Lab 2 hand-written Dockerfile.

**Lab 2 Dockerfile (you wrote):**
```dockerfile
FROM python:3.10-slim
WORKDIR /app
RUN pip install ... scikit-learn flask gunicorn google-cloud-storage ...
COPY train.py /app/train.py
COPY serve.py /app/serve.py
ENTRYPOINT ["python", "train.py"]
```

Notice what CPR adds: the model server entrypoint, handler configuration, and environment
variables. No Flask, no gunicorn — CPR's own model server handles HTTP.

In [59]:
# Inspect the container spec
container_spec = local_model.get_serving_container_spec()
print("Container Spec:")
print(f"  Image URI: {container_spec.image_uri}")
print(f"  Predict route: {container_spec.predict_route}")
print(f"  Health route: {container_spec.health_route}")

if container_spec.env:
    print(f"\nEnvironment Variables:")
    for env in container_spec.env:
        print(f"  {env.name} = {env.value}")

Container Spec:
  Image URI: us-central1-docker.pkg.dev/carty-470812/ml-containers/census-cpr:latest
  Predict route: /predict
  Health route: /health


### 3.3 Push Image to Artifact Registry

In [60]:
# Push the CPR image to Artifact Registry
local_model.push_image()
print(f"\n✅ CPR image pushed: {CPR_IMAGE_URI}")

/opt/homebrew/Cellar/python@3.13/3.13.5/Frameworks/Python.framework/Versions/3.13/lib/python3.13/subprocess.py:1023: RuntimeWarning: line buffering (buffering=1) isn't supported in binary mode, the default buffer size will be used
  self.stdin = io.open(p2cwrite, 'wb', bufsize)
/opt/homebrew/Cellar/python@3.13/3.13.5/Frameworks/Python.framework/Versions/3.13/lib/python3.13/subprocess.py:1029: RuntimeWarning: line buffering (buffering=1) isn't supported in binary mode, the default buffer size will be used
  self.stdout = io.open(c2pread, 'rb', bufsize)



✅ CPR image pushed: us-central1-docker.pkg.dev/carty-470812/ml-containers/census-cpr:latest


### 3.4 Upload Model to Vertex AI Model Registry

We upload the model with the CPR container as the serving image, pointing to the
model artifact in GCS. This creates a Model resource in the registry that's ready
to deploy to an endpoint.

In [61]:
# Upload model to Model Registry with CPR container
model = aiplatform.Model.upload(
    local_model=local_model,
    display_name=f"census-cpr-{TIMESTAMP}",
    artifact_uri=MODEL_DIR,
    description="Census income classifier served via Custom Prediction Routine (Mini-Lab A)",
)

print(f"✅ Model uploaded: {model.display_name}")
print(f"   Resource name: {model.resource_name}")

Creating Model
Create Model backing LRO: projects/873708835509/locations/us-central1/models/5740804195804512256/operations/7401546011966963712
Model created. Resource name: projects/873708835509/locations/us-central1/models/5740804195804512256@1
To use this Model in another session:
model = aiplatform.Model('projects/873708835509/locations/us-central1/models/5740804195804512256@1')
✅ Model uploaded: census-cpr-20260308-175839
   Resource name: projects/873708835509/locations/us-central1/models/5740804195804512256


### 3.5 Deploy to Endpoint

Deploy to a minimal machine type. We'll undeploy promptly after testing — the endpoint
charges per minute while a model is deployed.

In [62]:
# Create endpoint and deploy
endpoint = model.deploy(
    deployed_model_display_name=f"census-cpr-deployed-{TIMESTAMP}",
    machine_type="n1-standard-2",
    min_replica_count=1,
    max_replica_count=1,
)

print(f"\n✅ Model deployed to endpoint: {endpoint.display_name}")
print(f"   Resource name: {endpoint.resource_name}")

Creating Endpoint
Create Endpoint backing LRO: projects/873708835509/locations/us-central1/endpoints/7211882034292064256/operations/4351694270562828288
Endpoint created. Resource name: projects/873708835509/locations/us-central1/endpoints/7211882034292064256
To use this Endpoint in another session:
endpoint = aiplatform.Endpoint('projects/873708835509/locations/us-central1/endpoints/7211882034292064256')
Deploying model to Endpoint : projects/873708835509/locations/us-central1/endpoints/7211882034292064256
Deploy Endpoint model backing LRO: projects/873708835509/locations/us-central1/endpoints/7211882034292064256/operations/3997035799907401728
Endpoint model deployed. Resource name: projects/873708835509/locations/us-central1/endpoints/7211882034292064256

✅ Model deployed to endpoint: census-cpr-20260308-175839_endpoint
   Resource name: projects/873708835509/locations/us-central1/endpoints/7211882034292064256


### 3.6 Test Predictions

Use the same test instances from Lab 2 to verify predictions are sensible.

In [63]:
# Test instances — same profiles as Lab 2
test_instances = [
    {
        "age": 39, "workclass": " State-gov", "education": " Bachelors",
        "education_num": 13, "marital_status": " Never-married",
        "occupation": " Adm-clerical", "relationship": " Not-in-family",
        "race": " White", "sex": " Male", "capital_gain": 2174,
        "capital_loss": 0, "hours_per_week": 40, "native_country": " United-States"
    },
    {
        "age": 52, "workclass": " Self-emp-not-inc", "education": " HS-grad",
        "education_num": 9, "marital_status": " Married-civ-spouse",
        "occupation": " Exec-managerial", "relationship": " Husband",
        "race": " White", "sex": " Male", "capital_gain": 0,
        "capital_loss": 0, "hours_per_week": 45, "native_country": " United-States"
    },
    {
        "age": 28, "workclass": " Private", "education": " Masters",
        "education_num": 14, "marital_status": " Married-civ-spouse",
        "occupation": " Prof-specialty", "relationship": " Wife",
        "race": " White", "sex": " Female", "capital_gain": 0,
        "capital_loss": 0, "hours_per_week": 50, "native_country": " United-States"
    },
]

# Send prediction request
predictions = endpoint.predict(instances=test_instances)

print("=" * 60)
print("PREDICTION RESULTS (CPR Endpoint)")
print("=" * 60)

for i, (instance, pred) in enumerate(zip(test_instances, predictions.predictions), 1):
    print(f"\nExample {i}:")
    print(f"  Age: {instance['age']}, Education: {instance['education'].strip()}, "
          f"Occupation: {instance['occupation'].strip()}")

    # CPR postprocess returns structured dicts
    if isinstance(pred, dict):
        print(f"  Predicted: {pred.get('predicted_label', 'N/A')}")
        print(f"  Confidence: {pred.get('confidence', 0):.1%}")
        probs = pred.get('probabilities', {})
        print(f"  P(<=50K): {probs.get('<=50K', 0):.1%}  |  P(>50K): {probs.get('>50K', 0):.1%}")
    else:
        # Fallback if response format differs
        print(f"  Raw prediction: {pred}")

print("\n" + "=" * 60)

PREDICTION RESULTS (CPR Endpoint)

Example 1:
  Age: 39, Education: Bachelors, Occupation: Adm-clerical
  Predicted: <=50K
  Confidence: 99.6%
  P(<=50K): 99.6%  |  P(>50K): 0.4%

Example 2:
  Age: 52, Education: HS-grad, Occupation: Exec-managerial
  Predicted: <=50K
  Confidence: 94.8%
  P(<=50K): 94.8%  |  P(>50K): 5.2%

Example 3:
  Age: 28, Education: Masters, Occupation: Prof-specialty
  Predicted: <=50K
  Confidence: 83.2%
  P(<=50K): 83.2%  |  P(>50K): 16.8%



> **Compare to Lab 2 output:** In Lab 2, predictions came back as raw class labels (0 or 1).
> Here, the CPR `postprocess()` method enriches the output with human-readable labels,
> confidence scores, and per-class probabilities — all without changing the model itself.
> This is a direct benefit of CPR's structured pipeline: preprocessing and postprocessing
> are first-class concepts, not afterthoughts wedged into a Flask route.

---
## Part 4: Comparison, Reflection & Cleanup

### 4.1 Side-by-Side Comparison

| Dimension | Lab 2 (Flask Container) | Mini-Lab A (CPR) |
|-----------|------------------------|------------------|
| **Serving code** | `serve.py` (~45 lines) + Dockerfile (~15 lines) | `predictor.py` (~70 lines incl. docstrings) |
| **Files you wrote** | `serve.py`, `Dockerfile`, `requirements.txt` | `predictor.py`, `requirements.txt` |
| **HTTP server** | Flask (you wrote + configured) | Vertex AI model server (auto-generated) |
| **Health endpoint** | You wrote `/health` | Auto-generated |
| **Request parsing** | `request.get_json()` in Flask route | Default Handler deserializes |
| **Response formatting** | `jsonify()` in Flask route | Default Handler serializes |
| **Dockerfile** | Hand-written | Auto-generated by `build_cpr_model()` |
| **Container build** | `docker build` + `docker push` | `build_cpr_model()` + `push_image()` |
| **Model loading** | Custom `gsutil cp` + `joblib.load()` | `SklearnPredictor.load()` (inherited) |
| **Preprocessing** | Inside `/predict` route (mixed with routing) | Dedicated `preprocess()` method |
| **Postprocessing** | Inside `/predict` route (mixed with routing) | Dedicated `postprocess()` method |
| **Output format** | Raw predictions only | Labels + confidence + probabilities |
| **Error handling** | Manual try/except in route | Default Handler provides standard errors |

### Key Takeaway

CPR doesn't reduce the amount of ML logic you write — the preprocessing and inference code
is roughly the same. What it eliminates is the **undifferentiated plumbing**: HTTP server setup,
route definitions, health checks, request/response serialization, Dockerfile authoring.

That's why the exam signals "minimize code" or "minimize infrastructure maintenance" → CPR.
You're not writing less ML code; you're writing less non-ML code.

### 4.2 Exam Pattern Cheat Sheet

| Exam Signal | Points To |
|-------------|-----------|
| "Custom model + preprocessing + minimize code" | **CPR** |
| "scikit-learn/XGBoost + pre/postprocessing" | **CPR (SklearnPredictor)** |
| "Minimize infrastructure maintenance" + serving | **CPR** or prebuilt container |
| "Full control over serving logic" | Custom container (Lab 2 approach) |
| "Custom HTTP routing" or "non-JSON input" | Custom **Handler** (still CPR, with Handler override) |
| "No ML expertise" + predictions | Pre-trained API or AutoML |
| "Deploy only if accuracy > X" | `dsl.Condition` in Pipelines (Lab 10) |

### The Abstraction Spectrum

```
More managed ←──────────────────────────────────→ More control
Pre-trained  →  AutoML  →  CPR  →  Custom Container  →  Self-managed (GKE)
  APIs                     ▲
                      YOU ARE HERE
```

CPR sits at the sweet spot for most exam scenarios: enough customization for
preprocessing/postprocessing, but Vertex AI handles the infrastructure.

### 4.3 Reflection Questions

Answer these in your own words (edit this cell):

**1. When would you still choose Lab 2's approach (full custom container) over CPR?**

Think about what CPR can't do. The HTTP server, routing, and health checks are all managed by Vertex AI's default handler. So if you needed custom HTTP routing (like multiple endpoints beyond /predict), non-JSON input formats that the default handler can't parse, complex multi-model serving (loading two models and combining results in non-standard ways), or specific system-level dependencies that need to be baked into the container beyond pip packages — those would push you toward a full custom container. Also if you needed a non-Python serving stack entirely.

**2. Could you use CPR for your Lab 8 TensorFlow image classifier? What would change?**

Yes, but you'd subclass the raw Predictor base class instead of SklearnPredictor, since there's no TensorFlowPredictor convenience class. Your load() would use tf.saved_model.load() or tf.keras.models.load_model(). The preprocess() would handle the base64 decoding you built in Lab 8. The pattern is the same — CPR doesn't care what framework you use, it just needs the four methods.

**3. What would need to change if the census model needed to accept CSV input instead of JSON?**

This is where the Handler comes in. The default PredictionHandler expects JSON. For CSV, you'd write a custom Handler that deserializes CSV request bodies, then passes the parsed data to your Predictor. This is the one scenario where you customize the Handler — and it's still CPR, just with both a custom Predictor and a custom Handler.

**4. How does CPR relate to the TRANSFORM pattern from Lab 1 (train-serve skew prevention)?**

They're solving the same problem from different angles. The TRANSFORM clause in BigQuery ML bakes preprocessing into the model so it can't get out of sync between training and serving. CPR's preprocess() method serves the same purpose — it ensures the exact preprocessing logic runs at inference time. The risk is that if you change preprocessing in training but forget to update preprocess(), you get train-serve skew. The TRANSFORM pattern avoids this by making it impossible to forget. CPR requires discipline to keep them in sync, but gives you more flexibility.

### 4.4 Cleanup

Undeploy and delete resources to avoid ongoing charges.

In [ ]:
# Undeploy model from endpoint
endpoint.undeploy_all()
print("✅ Model undeployed")

# Delete endpoint
endpoint.delete()
print("✅ Endpoint deleted")

# Delete model from registry
model.delete()
print("✅ Model deleted from registry")

In [ ]:
# Delete GCS artifacts
!gsutil -m rm -r {BUCKET_URI}/mini-lab-a/

# Delete container images from Artifact Registry
!gcloud artifacts docker images delete {TRAIN_IMAGE_URI} --quiet 2>/dev/null
!gcloud artifacts docker images delete {CPR_IMAGE_URI} --quiet 2>/dev/null

print("✅ GCS artifacts and container images cleaned up")

---
## Mini-Lab A Summary

### What You Built
- Custom Prediction Routine using `SklearnPredictor` with preprocessing and postprocessing
- Auto-generated serving container via `LocalModel.build_cpr_model()`
- Live endpoint on Vertex AI returning structured predictions

### Key Learnings
1. **CPR eliminates HTTP plumbing, not ML logic** — same preprocessing, structured better
2. **`SklearnPredictor`** handles model loading; override `load()` to add extra artifacts
3. **`base_image` parameter** overrides CPR's default Python 3.7 — essential for version compatibility
4. **`build_cpr_model()`** generates Dockerfile + builds image in one call
5. **Exam pattern:** "custom model + preprocessing + minimize code" → CPR

### Exam Readiness
- [x] Can explain when to use CPR vs custom container vs prebuilt container
- [x] Know the Predictor interface: load → preprocess → predict → postprocess
- [x] Understand CPR's position on the abstraction spectrum
- [x] Recognize exam signals that point to CPR